# Случайное блуждание

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. "Лекция 5: Случайные блуждания на графах"
* Документация:
    * https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html
    * https://numpy.org/doc/stable/reference/generated/numpy.diag.html
    * https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
    * https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_power.html
    * https://numpy.org/doc/stable/reference/generated/numpy.allclose.html

## Вопросы для совместного обсуждения

1\. Обсудите понятие случайного блуждания на графах. 

## Задачи для самостоятельного решения

In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Tuple
from collections import Counter

# Настройки для красивого вывода numpy
np.set_printoptions(precision=4, suppress=True)

# Фиксируем seed для воспроизводимости результатов
np.random.seed(42)

<p class="task" id="1"></p>

1\. Загрузите граф карате-клуба. Получите матрицу смежности `A` этого графа. Получите на ее основе матрицу переходов `P` по следующему правилу:

$$\mathbf{P}=\mathbf{D}^{-1}\mathbf{A}$$

Продемонстрируйте, что выполняются условия (1) и (2).

$0 \le p_{ij} \le 1$ (1)

$\sum_j p_{ij}=1$    (2)

- [ ] Проверено на семинаре

In [2]:
G = nx.karate_club_graph()
N = G.number_of_nodes()

A = nx.adjacency_matrix(G).toarray()

degrees = A.sum(axis=1)


D_inv = np.diag(1.0 / degrees)

P = D_inv @ A
P


array([[0.    , 0.0952, 0.119 , ..., 0.0476, 0.    , 0.    ],
       [0.1379, 0.    , 0.2069, ..., 0.    , 0.    , 0.    ],
       [0.1515, 0.1818, 0.    , ..., 0.    , 0.0606, 0.    ],
       ...,
       [0.0952, 0.    , 0.    , ..., 0.    , 0.1905, 0.1905],
       [0.    , 0.    , 0.0526, ..., 0.1053, 0.    , 0.1316],
       [0.    , 0.    , 0.    , ..., 0.0833, 0.1042, 0.    ]],
      shape=(34, 34))

In [3]:
assert np.all((P >= 0) & (P <= 1)), "Ошибка: Вероятности должны быть от 0 до 1"

row_sums = P.sum(axis=1) # [N]
assert np.allclose(row_sums, 1.0), "Ошибка: Сумма вероятностей по строкам не равна 1"

print(f"Размерность матрицы P: {P.shape}")
print("Условия 1 и 2 успешно выполнены! Матрица P является стохастической.")

Размерность матрицы P: (34, 34)
Условия 1 и 2 успешно выполнены! Матрица P является стохастической.


<p class="task" id="2"></p>

2\. Создайте вектор начального состояния $\mathbf{p}^0 = [0, ..., 1]^T$. Получите стационарное состояние $\mathbf{p}^\infty$, используя итеративную процедуру

$\mathbf{p}^{t+1}=(\mathbf{P}^{\top})\mathbf{p}^t$ 

Процесс заканчивается, когда $||\mathbf{p}^{t+1} - \mathbf{p}^{t}|| < \epsilon $

Выведите полученный вектор стационарного состояния на экран.

- [ ] Проверено на семинаре

In [7]:
p_t = np.zeros((N, 1))
p_t[-1, 0] = 1.0

P_T = P.T

eps = 1e-6
max_iters = 1000
iters = 0

while iters < max_iters:
    p_new = P_T @ p_t
    
    if np.linalg.norm(p_new - p_t) < eps:
        p_t = p_new
        break
        
    p_t = p_new
    iters += 1

print(f"Процесс сошелся за {iters} итераций.")
print(f"Вектор стационарного состояния p_inf (первые 5 элементов из {p_t.shape[1]}):")
print(p_t[:5].flatten())

Процесс сошелся за 85 итераций.
Вектор стационарного состояния p_inf (первые 5 элементов из 1):
[0.0909 0.0628 0.0714 0.039  0.0173]


<p class="task" id="3"></p>

3\. Найдите матрицу перехода к стационарному состоянию $(\mathbf{P}^{\top})^\infty$ при помощи процедуры возведения матрицы в степень.

Докажите, что полученная матрица является матрицей стационарного состояния, т.е. $||(\mathbf{P}^{\top})^{\infty}  -(\mathbf{P}^{\top})(\mathbf{P}^{\top})^{\infty}|| <= \epsilon$

Выведите полученную матрицу на экран.

- [ ] Проверено на семинаре

In [14]:
n_power = 123456789987576464
P_T_inf = np.linalg.matrix_power(P_T, n_power)


lhs = P_T_inf
rhs = P_T @ P_T_inf
diff_norm = np.linalg.norm(lhs - rhs)

eps = 1e-6
print(f"Норма разности: {diff_norm:.2e}")
assert diff_norm <= eps, "Матрица не достигла стационарного состояния"

print("\nМатрица перехода к стационарному состоянию (P_T)^inf успешно вычислена.")

print("\nЛевый верхний угол 3x3 матрицы (P_T)^inf:")
print(P_T_inf[:3, :3])


print("\nПолная матрица(больше окна вывода моего IDE):")
P_T_inf

Норма разности: 1.26e-16

Матрица перехода к стационарному состоянию (P_T)^inf успешно вычислена.

Левый верхний угол 3x3 матрицы (P_T)^inf:
[[0.0126 0.0126 0.0126]
 [0.0087 0.0087 0.0087]
 [0.0099 0.0099 0.0099]]

Полная матрица(больше окна вывода моего IDE):


array([[0.0126, 0.0126, 0.0126, ..., 0.0126, 0.0126, 0.0126],
       [0.0087, 0.0087, 0.0087, ..., 0.0087, 0.0087, 0.0087],
       [0.0099, 0.0099, 0.0099, ..., 0.0099, 0.0099, 0.0099],
       ...,
       [0.0063, 0.0063, 0.0063, ..., 0.0063, 0.0063, 0.0063],
       [0.0114, 0.0114, 0.0114, ..., 0.0114, 0.0114, 0.0114],
       [0.0144, 0.0144, 0.0144, ..., 0.0144, 0.0144, 0.0144]],
      shape=(34, 34))

<p class="task" id="4"></p>

4\. Cоздайте вектор начального состояния $\mathbf{p}^0 = [0, ..., 1]^T $. Получите стационарное состояние $\mathbf{p}^\infty$, воспользовавшись полученной матрицей $(\mathbf{P}^{\top})^\infty$. Решите задачу двумя способами: при помощи матричного умножения и при помощи оператора индексации.

Используя функцию `np.allclose`, покажите, что векторы стационарных состояний, полученные двумя разными методами, совпадают (с точностью до тысячных).

- [ ] Проверено на семинаре

In [16]:
p_0 = np.zeros((N, 1))
p_0[-1, 0] = 1.0

p_inf_method1 = P_T_inf @ p_0 

p_inf_method2 = P_T_inf[:, -1].reshape(-1, 1)

are_close = np.allclose(p_inf_method1, p_inf_method2, atol=1e-3)
print(f"Результаты совпадают: {are_close}")

Результаты совпадают: True


<p class="task" id="5"></p>

5\. Напишите функцию `generate_walk`, которая принимает на вход граф `G`, начальный узел `node` и генерирует случайное блуждание длины `walk_len`, начинающееся с этого узла. Сгенерируйте несколько реализаций (не меньше 1000) случайных блужданий длины 10 с началом в узле 7 и выясните, в каком узле случайные блуждания заканчиваются чаще всего. Выведите номер этого узла на экран.

- [ ] Проверено на семинаре

In [20]:
def generate_walk(G: nx.Graph, node: int, walk_len: int) -> List[int]:
    walk = [node]
    current_node = node
    
    for _ in range(walk_len):
        neighbors = list(G.neighbors(current_node))
        
        if not neighbors:
            break
            
        next_node = np.random.choice(neighbors)
        walk.append(next_node)
        current_node = next_node
        
    return walk


In [36]:
start_node = 7
walk_length = 10
num_simulations = 10000

end_nodes = []

for _ in range(num_simulations):
    walk = generate_walk(G, start_node, walk_length)
    end_nodes.append(walk[-1])

end_node_counts = Counter(end_nodes)
most_common_node, count = end_node_counts.most_common(1)[0]

print(f"Проведено симуляций: {num_simulations}")
print(f"Стартовый узел: {start_node}")
print(f"Чаще всего блуждания заканчивались в узле: {most_common_node} (попаданий: {count})")

Проведено симуляций: 10000
Стартовый узел: 7
Чаще всего блуждания заканчивались в узле: 0 (попаданий: 1171)
